In [1]:
import os
os.chdir('..')
print(os.getcwd())

c:\Users\User\bangla-emotion-classifier


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata

from src.data_loader import load_bangla_emotion_data, filter_four_emotions
from src.utils import clean_english_text, clean_bangla_text, get_text_stats

print("All imports successful")

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


All imports successful


In [3]:
# Load data
train_df, val_df, test_df = load_bangla_emotion_data()

train_df = filter_four_emotions(train_df)
val_df   = filter_four_emotions(val_df)
test_df  = filter_four_emotions(test_df)

# Stats BEFORE cleaning — we'll compare this after cleaning
print("--- Stats BEFORE cleaning ---")
before_stats = get_text_stats(train_df, text_column='text')
for key, value in before_stats.items():
    print(f"  {key}: {value}")

Train size : 16000
Test size  : 2000
Val size   : 2000
--- Stats BEFORE cleaning ---
  total_samples: 14124
  empty_texts: 0
  avg_word_count: 18.99
  avg_char_count: 95.99


In [4]:
# Pick one raw sample
sample_text = train_df['text'].iloc[1]
print("BEFORE:")
print(repr(sample_text))
# repr() shows hidden characters like \n \t — useful for debugging

print("\nAFTER:")
cleaned = clean_english_text(sample_text)
print(repr(cleaned))

print("\nWhat changed:")
print(f"  Length before : {len(sample_text)} chars")
print(f"  Length after  : {len(cleaned)} chars")
print(f"  Words before  : {len(sample_text.split())}")
print(f"  Words after   : {len(cleaned.split())}")

BEFORE:
'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

AFTER:
'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

What changed:
  Length before : 108 chars
  Length after  : 108 chars
  Words before  : 21
  Words after   : 21


In [5]:
# Synthetic dirty text to test each cleaning step
dirty_samples = [
    "I feel SO happy today!!!   Check this out: http://example.com",
    "feeling <b>angry</b> about this...",
    "I'm   feeling    great    today",
    "FEELING JOYFUL AND BLESSED!!!",
]

print("Testing cleaning function on dirty samples:")
print("=" * 60)

for dirty in dirty_samples:
    cleaned = clean_english_text(dirty)
    print(f"\nBEFORE : {dirty}")
    print(f"AFTER  : {cleaned}")

Testing cleaning function on dirty samples:

BEFORE : I feel SO happy today!!!   Check this out: http://example.com
AFTER  : i feel so happy today check this out

BEFORE : feeling <b>angry</b> about this...
AFTER  : feeling angry about this

BEFORE : I'm   feeling    great    today
AFTER  : im feeling great today

BEFORE : FEELING JOYFUL AND BLESSED!!!
AFTER  : feeling joyful and blessed


In [6]:
# Apply cleaning to all splits
# We create a new column 'clean_text' — never overwrite the original 'text'
# Always keep raw data intact — golden rule of data preprocessing

train_df['clean_text'] = train_df['text'].apply(clean_english_text)
val_df['clean_text']   = val_df['text'].apply(clean_english_text)
test_df['clean_text']  = test_df['text'].apply(clean_english_text)

# Stats AFTER cleaning
print("--- Stats AFTER cleaning ---")
after_stats = get_text_stats(train_df, text_column='clean_text')
for key, value in after_stats.items():
    print(f"  {key}: {value}")

print("\n--- Comparison ---")
print(f"  avg_word_count : {before_stats['avg_word_count']} → {after_stats['avg_word_count']}")
print(f"  avg_char_count : {before_stats['avg_char_count']} → {after_stats['avg_char_count']}")
print(f"  empty_texts    : {before_stats['empty_texts']} → {after_stats['empty_texts']}")

--- Stats AFTER cleaning ---
  total_samples: 14124
  empty_texts: 0
  avg_word_count: 18.99
  avg_char_count: 95.98

--- Comparison ---
  avg_word_count : 18.99 → 18.99
  avg_char_count : 95.99 → 95.98
  empty_texts    : 0 → 0


In [7]:
# Check if any text became empty after cleaning
empty_mask = train_df['clean_text'].apply(lambda x: len(x.strip()) == 0)
empty_rows = train_df[empty_mask]

print(f"Empty texts after cleaning: {len(empty_rows)}")

if len(empty_rows) > 0:
    print("\nProblematic rows:")
    print(empty_rows[['text', 'clean_text', 'emotion']])
else:
    print("No texts were destroyed by cleaning.")
    print("Safe to proceed.")

Empty texts after cleaning: 0
No texts were destroyed by cleaning.
Safe to proceed.


In [8]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get unique classes
classes = np.array(['anger', 'fear', 'joy', 'sadness'])

# Compute weights
# 'balanced' means: weight = total_samples / (n_classes * class_count)
# Higher weight → model penalizes mistakes more for that class
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['emotion']
)

class_weights = dict(zip(classes, weights))

print("--- Class Weights ---")
for emotion, weight in class_weights.items():
    bar = '█' * int(weight * 10)
    print(f"  {emotion:10s} : {weight:.4f}  {bar}")

print("\nInterpretation:")
print("  Weight > 1.0 → underrepresented → model penalizes mistakes more")
print("  Weight < 1.0 → overrepresented  → model penalizes mistakes less")

--- Class Weights ---
  anger      : 1.6355  ████████████████
  fear       : 1.8229  ██████████████████
  joy        : 0.6585  ██████
  sadness    : 0.7568  ███████

Interpretation:
  Weight > 1.0 → underrepresented → model penalizes mistakes more
  Weight < 1.0 → overrepresented  → model penalizes mistakes less


In [9]:
import os

# Save cleaned dataframes
train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv',   index=False)
test_df.to_csv('data/processed/test.csv', index=False)

# Save class weights separately
import json
with open('data/processed/class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)

print("--- Saved files ---")
for fname in os.listdir('data/processed/'):
    fpath = f'data/processed/{fname}'
    size  = os.path.getsize(fpath)
    print(f"  {fname:30s} {size:>10,} bytes")

--- Saved files ---
  class_weights.json                    129 bytes
  test.csv                          360,002 bytes
  train.csv                       2,863,464 bytes
  val.csv                           348,448 bytes


In [10]:
# Reload from disk and verify
train_check = pd.read_csv('data/processed/train.csv')

print("--- Reloaded from disk ---")
print(f"Shape        : {train_check.shape}")
print(f"Columns      : {train_check.columns.tolist()}")
print(f"Emotions     : {train_check['emotion'].unique().tolist()}")
print(f"Empty texts  : {train_check['clean_text'].isnull().sum()}")
print()
print("--- Sample row ---")
print(train_check[['text', 'clean_text', 'emotion']].iloc[0])

print()

# Reload class weights
with open('data/processed/class_weights.json', 'r') as f:
    loaded_weights = json.load(f)

print("--- Class weights reloaded ---")
print(loaded_weights)

--- Reloaded from disk ---
Shape        : (14124, 4)
Columns      : ['text', 'label', 'emotion', 'clean_text']
Emotions     : ['sadness', 'anger', 'fear', 'joy']
Empty texts  : 0

--- Sample row ---
text          i didnt feel humiliated
clean_text    i didnt feel humiliated
emotion                       sadness
Name: 0, dtype: str

--- Class weights reloaded ---
{'anger': 1.635479388605836, 'fear': 1.8229220443985545, 'joy': 0.6585229392017904, 'sadness': 0.7567509644234891}


In [11]:
print("""
╔══════════════════════════════════════════════════════════╗
║         PHASE 2 SUMMARY — DATA PREPROCESSING            ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Cleaning applied                                        ║
║    ✓ Lowercasing                                         ║
║    ✓ URL removal                                         ║
║    ✓ HTML tag removal                                    ║
║    ✓ Punctuation removal                                 ║
║    ✓ Extra whitespace removal                            ║
║                                                          ║
║  Data integrity                                          ║
║    ✓ 0 empty texts after cleaning                        ║
║    ✓ Raw text preserved in 'text' column                 ║
║    ✓ Clean text saved in 'clean_text' column             ║
║                                                          ║
║  Class imbalance strategy                                ║
║    ✓ Class weights computed (not oversampling)           ║
║    anger   : 1.64                                        ║
║    fear    : 1.82                                        ║
║    joy     : 0.66                                        ║
║    sadness : 0.76                                        ║
║                                                          ║
║  Saved to disk                                           ║
║    ✓ data/processed/train.csv       (14,124 rows)        ║
║    ✓ data/processed/val.csv          (1,741 rows)        ║
║    ✓ data/processed/test.csv         (1,775 rows)        ║
║    ✓ data/processed/class_weights.json                   ║
║                                                          ║
║  Ready for Phase 3                                       ║
║    → TF-IDF + Logistic Regression baseline               ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         PHASE 2 SUMMARY — DATA PREPROCESSING            ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Cleaning applied                                        ║
║    ✓ Lowercasing                                         ║
║    ✓ URL removal                                         ║
║    ✓ HTML tag removal                                    ║
║    ✓ Punctuation removal                                 ║
║    ✓ Extra whitespace removal                            ║
║                                                          ║
║  Data integrity                                          ║
║    ✓ 0 empty texts after cleaning                        ║
║    ✓ Raw text preserved in 'text' column                 ║
║    ✓ Clean text saved in 'clean_text' column             ║
║                                                          ║
║  Class imbalance strat